In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import fastf1

fastf1.Cache.enable_cache("~/Documents/formula1/fastf1_cache")

all_data = []

for year in [2022,2023,2024,2025]:
    schedule = fastf1.get_event_schedule(year)
    schedule = schedule[
        schedule['EventFormat'].isin(['conventional','sprint_qualifying'])
    ]
    for _, event in schedule.iterrows():
        session = fastf1.get_session(year,event['RoundNumber'],'R')
        session.load()

        laps = session.laps.copy()

        laps['Year'] = year
        laps['Country'] = event['Country']
        laps['Circuit'] = event['Location']

        weather = session.weather_data.copy()

        laps = pd.merge_asof(
            laps.sort_values('Time'),
            weather.sort_values('Time'),
            on='Time',
            direction='nearest'
        )
        
        all_data.append(laps)

data = pd.concat(all_data,ignore_index=True)

data.to_csv("../data/RAWCombinedLapsWeather2022-2025.csv", index=False)

In [3]:
data = pd.read_csv("../data/RAWCombinedLapsWeather2022-2025.csv")

In [4]:
data.shape

(91315, 41)

In [5]:
data.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate', 'Year', 'Country', 'Circuit',
       'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp',
       'WindDirection', 'WindSpeed'],
      dtype='object')

In [6]:
data = data.sort_values(['Circuit','Year','Driver','LapNumber'])

### Filling in missing LapTimes

In [7]:
data['LapTime'] = pd.to_timedelta(data['LapTime'])          # converting LapTime to timedelta dtype

In [8]:
data['Sector1Time'] = pd.to_timedelta(data['Sector1Time'])          # converting Sector1Time to timedelta dtype
data['Sector2Time'] = pd.to_timedelta(data['Sector2Time'])          # converting Sector2Time to timedelta dtype
data['Sector3Time'] = pd.to_timedelta(data['Sector3Time'])          # converting Sector3Time to timedelta dtype


In [9]:
sector_cols = ["Sector1Time", "Sector2Time", "Sector3Time"]  # Filling in missing Laptimes with added sector times (only if all sector times available and if laptime is missing)

mask = (
    data["LapTime"].isna()
    & data[sector_cols].notna().all(axis=1)
)

data.loc[mask, "LapTime"] = data.loc[mask, sector_cols].sum(axis=1)

In [10]:
data['LapTimeSeconds'] = data['LapTime'].dt.total_seconds() # making laptimeseconds col (target)

data['SectorSum'] = data[sector_cols].sum(axis=1).dt.total_seconds()
diff = (data["LapTimeSeconds"] - data["SectorSum"]).abs()
diff.describe() 

Used this to check if we could do the method above to fill in missing laptimes -> was conclusive enough to prove we could

In [11]:
data[data['LapTime'].isna()][['Driver','Circuit','Year','LapNumber','LapTime','Sector1Time', 'Sector2Time', 'Sector3Time','LapTimeSeconds']] # 1469 -> 352 missing laptimes (by adding sector times)

,Driver,Circuit,Year,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,LapTimeSeconds
16697,BOT,Austin,2022,17.0,NaT,NaT,NaT,NaT,NaN
16787,STR,Austin,2022,22.0,NaT,NaT,NaT,NaT,NaN
58341,HAM,Austin,2024,2.0,NaT,NaT,NaT,NaT,NaN
84744,SAI,Austin,2025,6.0,NaT,NaT,NaT,NaT,NaN
7050,MAG,Baku,2022,32.0,NaT,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...
31570,TSU,Zandvoort,2023,64.0,NaT,NaT,0 days 00:00:32.331000,0 days 00:00:36.859000,NaN
31576,VER,Zandvoort,2023,65.0,NaT,NaT,0 days 00:00:49.067000,0 days 00:00:40.051000,NaN
31547,ZHO,Zandvoort,2023,63.0,NaT,NaT,NaT,NaT,NaN
80549,HAM,Zandvoort,2025,23.0,NaT,NaT,NaT,NaT,NaN


In [12]:
data.shape

(91315, 42)

## CREATING FEATURES

In [13]:
data['InLap'] = data['PitInTime'].notna().astype(int)  # making InLap and OutLap features
data['OutLap'] = data['PitOutTime'].notna().astype(int)

In [14]:
data['Circuit'].unique()

array(['Austin', 'Baku', 'Barcelona', 'Budapest', 'Imola', 'Jeddah',
       'Las Vegas', 'Le Castellet', 'Lusail', 'Marina Bay', 'Melbourne',
       'Mexico City', 'Miami', 'Miami Gardens', 'Monaco', 'Montréal',
       'Monza', 'Sakhir', 'Shanghai', 'Silverstone', 'Spa-Francorchamps',
       'Spielberg', 'Suzuka', 'São Paulo', 'Yas Island', 'Zandvoort'],
      dtype=object)

In [15]:
data['Circuit'] = data['Circuit'].replace({ # fixing miami bug
    'Miami Gardens' : 'Miami'
})

In [16]:
data = data.sort_values(['Year','Circuit','Driver','LapNumber']) # sorting data in a more perferable format (already done but broooo)

In [17]:
data['TotalLaps'] = (
    data.groupby(['Year','Circuit'])['LapNumber'].transform('max')
)

data['LapsRemaining'] = data['TotalLaps'] - data['LapNumber'] # creating LapsRemaining feature

In [18]:
# Dealing with track status
priority = [5,4,6,7,2,1] # prioritising track status by redflag, safetycar, vsc deployed, vsc ending, yellow flag, track clear

def simplify_track_status(status):
    status = str(status)

    for code in priority:
        if str(code) in status:
            return code

    return 0 # unknown

data['TrackStatusSimple'] = data['TrackStatus'].apply(simplify_track_status)

# Below if needed (adding prev lap time and rolling lap time)

# Adding prev lap time and rolling lap time
group = data.groupby(['Year','Circuit','Driver'])

data['PrevLapNumber'] = group['LapNumber'].shift(1)
data['PreviousLapTime'] = group['LapTimeSeconds'].shift(1)
data['PreviousInLap'] = group['InLap'].shift(1)
data['PreviousOutLap'] = group['OutLap'].shift(1)

data['LapGap'] = data['LapNumber'] - data['PrevLapNumber']

# If the previous row isn't exactly one lap earlier,
# don't use its information.
valid_previous = data['LapGap'] == 1

data.loc[~valid_previous, 'PreviousLapTime'] = pd.NA
data.loc[~valid_previous, 'PreviousInLap'] = pd.NA
data.loc[~valid_previous, 'PreviousOutLap'] = pd.NA

# create lag2,lag3

data['LapTimeLag2'] = group['LapTimeSeconds'].shift(2)
data['LapTimeLag3'] = group['LapTimeSeconds'].shift(3)

data['LapNumberLag2'] = group['LapNumber'].shift(2)
data['LapNumberLag3'] = group['LapNumber'].shift(3)

data['InLapLag2'] = group['InLap'].shift(2)
data['OutLapLag2'] = group['OutLap'].shift(2)

data['InLapLag3'] = group['InLap'].shift(3)
data['OutLapLag3'] = group['OutLap'].shift(3)

# validating
gap2 = data['LapNumber'] - data['LapNumberLag2']
gap3 = data['LapNumber'] - data['LapNumberLag3']

valid_lag2 = gap2 == 2
valid_lag3 = gap3 == 3

data.loc[~valid_lag2, [
    'LapTimeLag2',
    'InLapLag2',
    'OutLapLag2'
]] = pd.NA

data.loc[~valid_lag3, [
    'LapTimeLag3',
    'InLapLag3',
    'OutLapLag3'
]] = pd.NA


data['RollingLapTime_3'] = (
    data[['PreviousLapTime', 'LapTimeLag2', 'LapTimeLag3']]
    .mean(axis=1)
)


In [19]:
features = [
    'Driver',
    'Circuit',
    'Year',
    'LapNumber',
    'LapsRemaining',
    'Compound',
    'TyreLife',
    'TrackStatusSimple',
    'Position',
    'AirTemp',
    'Humidity',
    'Pressure',
    'TrackTemp',
    'WindDirection',
    'WindSpeed',
    'Stint',
    'Rainfall',
    'InLap',
    'OutLap',
]

## DEALING WITH MISSING DATA

In [20]:
clean_data = data[features + ['LapTimeSeconds']].copy() 

In [21]:
clean_data.isna().sum()

Driver                 0
Circuit                0
Year                   0
LapNumber              0
LapsRemaining          0
Compound             481
TyreLife             630
TrackStatusSimple      0
Position             124
AirTemp                0
Humidity               0
Pressure               0
TrackTemp              0
WindDirection          0
WindSpeed              0
Stint                354
Rainfall               0
InLap                  0
OutLap                 0
LapTimeSeconds       352
dtype: int64

In [22]:
clean_data[clean_data['TyreLife'].isna()][['Driver','Circuit','Year']].drop_duplicates() # interchange between 'Compound', TyreLife, Stint to see all the missings

,Driver,Circuit,Year
5656,ALB,Monaco,2022
5702,ALO,Monaco,2022
5689,BOT,Monaco,2022
5727,GAS,Monaco,2022
5585,HAM,Monaco,2022
5738,LAT,Monaco,2022
5638,LEC,Monaco,2022
5640,MSC,Monaco,2022
5623,NOR,Monaco,2022
5705,OCO,Monaco,2022


Mostly from miami and spa 25, will deal with those first (as have delt with before)

missing data collected from https://www.formula1db.com/races/2025-miami-grand-prix/stats/race/tyres

**STINT MISSING DATA**

In [27]:
# Dealing with Miami - all missing data is from the first stint
clean_data.loc[
    (clean_data['Year'] == 2025) &
    (clean_data['Circuit'] == 'Miami') &
    (clean_data['Stint'].isna()),
    'Stint'
] = 1

**COMPOUND MISSING DATA**

In [23]:
# Dealing with Miami - missing hard and medium drivers from first stint

missing_hard_drivers_miami25 = ['ALO', 'RUS', 'HAM', 'LAW', 'HUL', 'BEA', 'GAS']

clean_data.loc[
    (clean_data['Year'] == 2025) &
    (clean_data['Circuit'] == 'Miami') &
    (clean_data['Driver'].isin(missing_hard_drivers_miami25)) &
    (clean_data['Compound'].isna()),
    'Compound'
] = 'HARD'

missing_medium_drivers_miami25 = ['ALB', 'ANT', 'LEC', 'NOR', 'PIA', 'SAI', 'TSU', 'VER']

clean_data.loc[
    (clean_data['Year'] == 2025) &
    (clean_data['Circuit'] == 'Miami') &
    (clean_data['Driver'].isin(missing_medium_drivers_miami25)) &
    (clean_data['Compound'].isna()),
    'Compound'
] = 'MEDIUM'

In [24]:
# Dealing with Spa 25 - missing medium drivers

missing_medium_drivers_spa25 = ['ALO', 'ANT', 'HUL', 'COL']

clean_data.loc[
    (clean_data['Year'] == 2025) &
    (clean_data['Circuit'] == 'Spa-Francorchamps') &
    (clean_data['Driver'].isin(missing_medium_drivers_spa25)) &
    (clean_data['Compound'].isna()),
    'Compound'
] = 'MEDIUM'

In [25]:
# Dealing with Montreal 23 - TSU missing hard
clean_data.loc[
    (clean_data['Year'] == 2023) &
    (clean_data['Circuit'] == 'Montréal') &
    (clean_data['Driver'] == 'TSU') &
    (clean_data['Compound'].isna()),
    'Compound'  
] = 'HARD'

In [26]:
# Dealing with Monaco 22 - STR missing hard
clean_data.loc[
    (clean_data['Year'] == 2022) &
    (clean_data['Circuit'] == 'Monaco') &
    (clean_data['Driver'] == 'STR') &
    (clean_data['Compound'].isna()),
    'Compound'  
] = 'HARD'

**TyreLife MISSING DATA**

In [28]:
# Miami 25 - All drivers started on fresh tyres, apart from SAI and ALO, start on two lap older used tyres (all for first stint as well so can just use lapnumber for tyrelife)

mask = (
    (clean_data['Year'] == 2025) &
    (clean_data['Circuit'] == 'Miami') &
    (clean_data['Stint'] == 1)
)

clean_data.loc[mask, 'TyreLife'] = clean_data.loc[mask, 'LapNumber']

used_tyre_miami25 = ['SAI', 'ALO']

clean_data.loc[
    mask & clean_data['Driver'].isin(used_tyre_miami25),
    'TyreLife'
] = clean_data.loc[
    mask & clean_data['Driver'].isin(used_tyre_miami25),
    'LapNumber'
] + 2

In [29]:
# Spa 25 - All drivers with missing data in stint 3 started with fresh tyres

mask = (
    (clean_data['Year'] == 2025) &
    (clean_data['Circuit'] == 'Spa-Francorchamps') &
    (clean_data['Stint'] == 3) &
    (clean_data['Driver'].isin(['ALO', 'ANT', 'COL', 'HUL','SAI'])) &
    (clean_data['TyreLife'].isna())
)

stint_start = (
    clean_data[
        (clean_data['Year'] == 2025) &
        (clean_data['Circuit'] == 'Spa-Francorchamps') &
        (clean_data['Stint'] == 3) &
        (clean_data['Driver'].isin(['ALO', 'ANT', 'COL', 'HUL','SAI']))
    ]
    .groupby('Driver')['LapNumber']
    .transform('min')
)

clean_data.loc[mask, 'TyreLife'] = (
    clean_data.loc[mask, 'LapNumber'] - stint_start.loc[mask] + 1
)

In [30]:
# Melb 25 - BEA was on new mediums
mask = (
    (clean_data['Year'] == 2025) &
    (clean_data['Circuit'] == 'Melbourne') & 
    (clean_data['Stint'] == 5) & 
    (clean_data['Driver'] == 'BEA') &
    (clean_data['TyreLife'].isna())
)

stint_start = (
    clean_data[
        (clean_data['Year'] == 2025) &
        (clean_data['Circuit'] == 'Melbourne') & 
        (clean_data['Stint'] == 5) & 
        (clean_data['Driver'] == 'BEA')   
    ]
    .groupby('Driver')['LapNumber']
    .transform('min')
)

clean_data.loc[mask,'TyreLife'] = (
    clean_data.loc[mask, 'LapNumber'] - stint_start.loc[mask] + 1
)

In [31]:
# Montreal 23 - tsunoda was on fresh hard tyres
mask = (
    (clean_data['Year'] == 2023) &
    (clean_data['Circuit'] == 'Montréal') & 
    (clean_data['Stint'] == 3) & 
    (clean_data['Driver'] == 'TSU') &
    (clean_data['TyreLife'].isna())
)

stint_start = (
    clean_data[
        (clean_data['Year'] == 2023) &
        (clean_data['Circuit'] == 'Montréal') & 
        (clean_data['Stint'] == 3) & 
        (clean_data['Driver'] == 'TSU')   
    ]
    .groupby('Driver')['LapNumber']
    .transform('min')
)

clean_data.loc[mask,'TyreLife'] = (
    clean_data.loc[mask, 'LapNumber'] - stint_start.loc[mask] + 1
)

In [32]:
# Monaco 2022 - Stint 2 drivers, ALB, ALO, BOT, HAM, LEC, NOR, OCO, PER, RIC, RUS, SAI, VER, ZHO, all on fresh tyres
mask = (
    (clean_data['Year'] == 2022) &
    (clean_data['Circuit'] == 'Monaco') &
    (clean_data['Stint'] == 2) &
    (clean_data['Driver'].isin(['ALB', 'ALO', 'BOT', 'HAM','LEC','NOR','OCO','PER','RIC','RUS','SAI','VER','ZHO'])) &
    (clean_data['TyreLife'].isna())
)

stint_start = (
    clean_data[
        (clean_data['Year'] == 2022) &
        (clean_data['Circuit'] == 'Monaco') &
        (clean_data['Stint'] == 2) &
        (clean_data['Driver'].isin(['ALB', 'ALO', 'BOT', 'HAM','LEC','NOR','OCO','PER','RIC','RUS','SAI','VER','ZHO']))
    ]
    .groupby('Driver')['LapNumber']
    .transform('min')
)

clean_data.loc[mask, 'TyreLife'] = (
    clean_data.loc[mask, 'LapNumber'] - stint_start.loc[mask] + 1
)

In [33]:
# Monaco 2022 - Stint 3 drivers, GAS, MSC, TSU, all on fresh tyres
mask = (
    (clean_data['Year'] == 2022) &
    (clean_data['Circuit'] == 'Monaco') &
    (clean_data['Stint'] == 3) &
    (clean_data['Driver'].isin(['GAS','MSC','TSU'])) &
    (clean_data['TyreLife'].isna())
)

stint_start = (
    clean_data[
        (clean_data['Year'] == 2022) &
        (clean_data['Circuit'] == 'Monaco') &
        (clean_data['Stint'] == 3) &
        (clean_data['Driver'].isin(['GAS','MSC','TSU']))
    ]
    .groupby('Driver')['LapNumber']
    .transform('min')
)

clean_data.loc[mask, 'TyreLife'] = (
    clean_data.loc[mask, 'LapNumber'] - stint_start.loc[mask] + 1
)

In [34]:
# Monaco 2022 - Stint 4 drivers, LAT, fresh tyres
mask = (
    (clean_data['Year'] == 2022) &
    (clean_data['Circuit'] == 'Monaco') &
    (clean_data['Stint'] == 4) &
    (clean_data['Driver'].isin(['LAT'])) &
    (clean_data['TyreLife'].isna())
)

stint_start = (
    clean_data[
        (clean_data['Year'] == 2022) &
        (clean_data['Circuit'] == 'Monaco') &
        (clean_data['Stint'] == 4) &
        (clean_data['Driver'].isin(['LAT']))
    ]
    .groupby('Driver')['LapNumber']
    .transform('min')
)

clean_data.loc[mask, 'TyreLife'] = (
    clean_data.loc[mask, 'LapNumber'] - stint_start.loc[mask] + 1
)

clean_data[
    (clean_data['Circuit'] == 'Monaco') &
    (clean_data['Year'] == 2022) & 
    (clean_data['TyreLife'].isna())
][['Driver','LapNumber','Compound','TyreLife','Stint']].sort_values(['Stint','Driver'])

In [35]:
clean_data.isna().sum()

Driver                 0
Circuit                0
Year                   0
LapNumber              0
LapsRemaining          0
Compound               0
TyreLife               0
TrackStatusSimple      0
Position             124
AirTemp                0
Humidity               0
Pressure               0
TrackTemp              0
WindDirection          0
WindSpeed              0
Stint                  0
Rainfall               0
InLap                  0
OutLap                 0
LapTimeSeconds       352
dtype: int64

**Position Missing Values**

From analysing, its because they retired from race, so lets just eliminate

In [36]:
clean_data = clean_data[ 
    clean_data['Position'].notna()   # getting rid of rest of Nan LapTimes (as there is nothing we can do)
]

clean_data["PrevPosition"] = (
    clean_data.groupby(["Year", "Circuit", "Driver"])["Position"]
        .shift(1)
)
clean_data["NextPosition"] = (
    clean_data.groupby(["Year", "Circuit", "Driver"])["Position"]
        .shift(-1)
)
missing_position = clean_data[clean_data["Position"].isna()]
print(
    missing_position[
        ["Year", "Circuit", "Driver", "LapNumber",
         "PrevPosition", "NextPosition"]
    ].iloc[50:100]
)


In [44]:
# Dont need to run if don't have PrevioiusLaptime, ... RollingLapTime_3
clean_data = clean_data.dropna(subset=['PreviousLapTime','PreviousInLap','PreviousOutLap','RollingLapTime_3'])

KeyError: ['PreviousLapTime', 'PreviousInLap', 'PreviousOutLap', 'RollingLapTime_3']

## DEALING WITH OUTLIERS DATA

In [37]:
clean_data['LapTimeSeconds'].describe()

count    90963.000000
mean        93.290038
std         35.466178
min         67.694000
25%         82.258500
50%         90.695000
75%         99.097500
max       2526.253000
Name: LapTimeSeconds, dtype: float64

In [38]:
clean_data['LapTimeSeconds'].quantile([0.01,0.05,0.5,0.95,0.99])

0.01     70.71400
0.05     75.38120
0.50     90.69500
0.95    121.45230
0.99    156.51514
Name: LapTimeSeconds, dtype: float64

In [39]:
clean_data[clean_data['LapTimeSeconds']>200][['Year','Circuit','Driver','LapNumber','LapTimeSeconds','InLap','OutLap']].sort_values('LapTimeSeconds',ascending=False)

,Year,Circuit,Driver,LapNumber,LapTimeSeconds,InLap,OutLap
45357,2024,Monaco,ZHO,1.0,2526.253,1,0
45356,2024,Monaco,BOT,1.0,2521.748,1,0
45355,2024,Monaco,SAR,1.0,2516.858,1,0
45354,2024,Monaco,ALO,1.0,2513.184,1,0
45353,2024,Monaco,RIC,1.0,2507.607,1,0
45352,2024,Monaco,STR,1.0,2502.648,1,0
45351,2024,Monaco,GAS,1.0,2497.695,1,0
45350,2024,Monaco,ALB,1.0,2494.817,1,0
45349,2024,Monaco,TSU,1.0,2488.609,1,0
45348,2024,Monaco,HAM,1.0,2483.804,1,0


In [40]:
#Evidently something is wrong as 2400-2500 seconds for a lap is absurd (probably a red flag of sorts), so will remove
clean_data = clean_data[
    ~(
        (clean_data['Year'] == 2024) & 
        (clean_data['Circuit'] == 'Monaco') &
        (clean_data['LapNumber'] == 1) & 
        (clean_data['LapTimeSeconds'] > 2000)
    )
]

## Extras

In [41]:
clean_data.shape

(91175, 20)

In [ ]:
# OPTIONAL -> do if you are going to do the regressors below, dont do if creating sequences for LSTMS/rnn's in a min

clean_data = clean_data[ 
    clean_data['LapTime'].notna()   # getting rid of rest of Nan LapTimes (as there is nothing we can do)
]

In [50]:
# SAVING to data (to be used by eventual lstm and thingos) <--- note 
clean_data.to_csv('../data/Cleaned_data2022-2025.csv',index=False)